In [ ]:
!pip -q install kaggle scikit-learn pandas pillow tqdm onnx onnxruntime

import os
import json
import random
import hashlib
import copy
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    f1_score
)

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 25
PATIENCE = 5
NUM_WORKERS = 2

PROJECT_DIR = Path("/content/driver_model_project")
DOWNLOAD_DIR = PROJECT_DIR / "downloads"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Maximum epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu
Image size: 256
Maximum epochs: 25
Early stopping patience: 5


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Install/update Kaggle API
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"],
    check=True
)

# Kaggle API credentials (your existing method)
os.environ["KAGGLE_USERNAME"] = "devkumbharvadia"
os.environ["KAGGLE_KEY"] = "KGAT_f3650be0cda56ffc7ba129c12b02eae9"

# Locate Kaggle executable installed in this Python environment
import shutil

KAGGLE_EXE = shutil.which("kaggle")

if KAGGLE_EXE is None:
    KAGGLE_EXE = str(Path(sys.executable).parent / "kaggle")

if not Path(KAGGLE_EXE).exists():
    raise FileNotFoundError("Kaggle executable was not found after installation.")

print("Kaggle executable:", KAGGLE_EXE)

# Dataset download directory
DATASET_DIR = DOWNLOAD_DIR / "alternative_driver"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Download and extract
result = subprocess.run(
    [
        KAGGLE_EXE,
        "datasets", "download",
        "-d", "yehiahassanain/new-driver-behaviour22222",
        "-p", str(DATASET_DIR),
        "--unzip"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("Kaggle error:")
    print(result.stderr)
    raise RuntimeError("Dataset download failed.")

# Locate training directory
train_candidates = list(DATASET_DIR.rglob("train"))

TRAIN_DIR = next(
    (
        path for path in train_candidates
        if all((path / f"c{i}").is_dir() for i in range(9))
    ),
    None
)

if TRAIN_DIR is None:
    raise FileNotFoundError(
        f"Could not find train/c0 through train/c8 under {DATASET_DIR}"
    )

print("\nTraining folder:", TRAIN_DIR)
print("Original Test folder:", TRAIN_DIR.parent / "test")

for class_dir in sorted(TRAIN_DIR.iterdir()):
    if class_dir.is_dir():
        print(f"{class_dir.name}: {len(list(class_dir.iterdir()))} files")

Kaggle executable: /usr/local/bin/kaggle
Dataset URL: https://www.kaggle.com/datasets/yehiahassanain/new-driver-behaviour22222
License(s): unknown



Training folder: /content/driver_model_project/downloads/alternative_driver/New_Driver_Behaviour2/train
Original Test folder: /content/driver_model_project/downloads/alternative_driver/New_Driver_Behaviour2/test
c0: 3109 files
c1: 2767 files
c2: 2797 files
c3: 2926 files
c4: 2736 files
c5: 2772 files
c6: 2805 files
c7: 2472 files
c8: 2279 files


In [ ]:

import os
import sys
import json
import random
import hashlib
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    f1_score
)

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Training configuration
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 25
PATIENCE = 5
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PIN_MEMORY = DEVICE.type == "cuda"

# Existing project directories
PROJECT_DIR = Path("/content/driver_model_project")
DOWNLOAD_DIR = PROJECT_DIR / "downloads"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Output directory:", OUTPUT_DIR)


Device: cpu
Image size: 256
Batch size: 32
Output directory: /content/driver_model_project/outputs


In [ ]:

# ============================================================
# DISCOVER DATASET CLASSES
# ============================================================

CLASS_DIRS = sorted(
    [
        path for path in TRAIN_DIR.iterdir()
        if path.is_dir() and path.name.startswith("c")
    ],
    key=lambda path: int(path.name[1:])
)

CLASS_NAMES = [path.name for path in CLASS_DIRS]

NUM_CLASSES = len(CLASS_NAMES)

EXPECTED_CLASSES = [f"c{i}" for i in range(9)]

if CLASS_NAMES != EXPECTED_CLASSES:
    raise RuntimeError(
        f"Expected {EXPECTED_CLASSES}, found {CLASS_NAMES}"
    )

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {
    index: name
    for name, index in CLASS_TO_IDX.items()
}

print("Number of classes:", NUM_CLASSES)
print("Class mapping:", CLASS_TO_IDX)

for class_dir in CLASS_DIRS:
    print(
        f"{class_dir.name}: "
        f"{sum(1 for p in class_dir.rglob('*') if p.is_file())} files"
    )


Number of classes: 9
Class mapping: {'c0': 0, 'c1': 1, 'c2': 2, 'c3': 3, 'c4': 4, 'c5': 5, 'c6': 6, 'c7': 7, 'c8': 8}
c0: 3109 files
c1: 2767 files
c2: 2797 files
c3: 2926 files
c4: 2736 files
c5: 2772 files
c6: 2805 files
c7: 2472 files
c8: 2279 files


In [ ]:

# ============================================================
# SCAN ALL IMAGES
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

records = []

invalid_images = []

for class_name in CLASS_NAMES:

    class_dir = TRAIN_DIR / class_name

    image_paths = [
        path
        for path in class_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    print(f"\nScanning {class_name}: {len(image_paths)} images")

    for image_path in tqdm(image_paths, leave=False):

        try:
            # Verify image can be opened.
            with Image.open(image_path) as image:
                image.verify()

            # Calculate exact file hash.
            sha256 = hashlib.sha256()

            with open(image_path, "rb") as file:
                for chunk in iter(
                    lambda: file.read(1024 * 1024),
                    b""
                ):
                    sha256.update(chunk)

            records.append({
                "path": str(image_path),
                "label": CLASS_TO_IDX[class_name],
                "class_name": class_name,
                "group": sha256.hexdigest()
            })

        except Exception as error:

            invalid_images.append({
                "path": str(image_path),
                "error": str(error)
            })

print("\nValid images:", len(records))
print("Invalid images:", len(invalid_images))


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError("No valid images were found.")

# Save invalid image report.
pd.DataFrame(invalid_images).to_csv(
    OUTPUT_DIR / "invalid_images.csv",
    index=False
)


# ============================================================
# FIND CONFLICTING EXACT DUPLICATES
# ============================================================

label_counts = df.groupby("group")["label"].nunique()

conflicting_hashes = label_counts[
    label_counts > 1
].index

conflict_df = df[
    df["group"].isin(conflicting_hashes)
].copy()

print(
    "\nConflicting duplicate image groups:",
    len(conflicting_hashes)
)

if len(conflicting_hashes):

    print("Excluding ambiguous exact duplicate images.")

    conflict_df.to_csv(
        OUTPUT_DIR / "conflicting_duplicates.csv",
        index=False
    )

    # Remove every copy of an ambiguous image.
    df = df[
        ~df["group"].isin(conflicting_hashes)
    ].copy()


# ============================================================
# REMOVE SAME-CLASS EXACT DUPLICATES
# ============================================================

before_deduplication = len(df)

df = (
    df.sort_values("path")
    .drop_duplicates(
        subset="group",
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Same-class duplicate copies removed:",
    before_deduplication - len(df)
)


# ============================================================
# VERIFY ALL CLASSES REMAIN
# ============================================================

remaining_classes = set(df["class_name"])

missing_classes = set(CLASS_NAMES) - remaining_classes

if missing_classes:
    raise RuntimeError(
        f"Classes missing after duplicate handling: {missing_classes}"
    )

class_counts = (
    df.groupby("class_name")
    .size()
    .reindex(CLASS_NAMES, fill_value=0)
)

print("\nFINAL DATASET DISTRIBUTION")

display(class_counts.rename("image_count").to_frame())

if (class_counts < 5).any():
    raise RuntimeError(
        "At least one class has fewer than five unique images."
    )

print("\nDataset preparation successful.")
print("Total unique images:", len(df))
print("Independent classes:", NUM_CLASSES)



Scanning c0: 3109 images


  0%|          | 0/3109 [00:00<?, ?it/s]


Scanning c1: 2767 images


  0%|          | 0/2767 [00:00<?, ?it/s]


Scanning c2: 2797 images


  0%|          | 0/2797 [00:00<?, ?it/s]

... (11 outputs left out)

  0%|          | 0/2279 [00:00<?, ?it/s]


Valid images: 24663
Invalid images: 0

Conflicting duplicate image groups: 1
Excluding ambiguous exact duplicate images.
Same-class duplicate copies removed: 0

FINAL DATASET DISTRIBUTION


,image_count
class_name,
c0,3109
c1,2767
c2,2797
c3,2926
c4,2736
c5,2772
c6,2804
c7,2472
c8,2278



Dataset preparation successful.
Total unique images: 24661
Independent classes: 9


In [ ]:

# ============================================================
# STRATIFIED GROUP SPLIT
# ============================================================

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

train_indices, val_indices = next(
    splitter.split(
        df,
        y=df["label"],
        groups=df["group"]
    )
)

train_df = (
    df.iloc[train_indices]
    .reset_index(drop=True)
)

val_df = (
    df.iloc[val_indices]
    .reset_index(drop=True)
)


# ============================================================
# VERIFY SPLIT
# ============================================================

assert set(train_df["group"]).isdisjoint(
    set(val_df["group"])
), "Duplicate leakage detected."

assert set(train_df["label"].unique()) == set(range(NUM_CLASSES))

assert set(val_df["label"].unique()) == set(range(NUM_CLASSES))

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

print("\nTraining distribution:")

display(
    train_df.groupby("class_name")
    .size()
    .reindex(CLASS_NAMES)
    .rename("training_images")
    .to_frame()
)

print("\nValidation distribution:")

display(
    val_df.groupby("class_name")
    .size()
    .reindex(CLASS_NAMES)
    .rename("validation_images")
    .to_frame()
)


# Save split files.
train_df.to_csv(
    OUTPUT_DIR / "train_split.csv",
    index=False
)

val_df.to_csv(
    OUTPUT_DIR / "validation_split.csv",
    index=False
)

# Save class mapping.
with open(
    OUTPUT_DIR / "class_mapping.json",
    "w"
) as file:

    json.dump({
        "class_names": CLASS_NAMES,
        "class_to_idx": CLASS_TO_IDX,
        "num_classes": NUM_CLASSES
    }, file, indent=4)

print("\nTrain/validation split completed.")


Training images: 19728
Validation images: 4933

Training distribution:


,training_images
class_name,
c0,2509
c1,2194
c2,2232
c3,2369
c4,2181
c5,2202
c6,2259
c7,1984
c8,1798



Validation distribution:


,validation_images
class_name,
c0,600
c1,573
c2,565
c3,557
c4,555
c5,570
c6,545
c7,488
c8,480



Train/validation split completed.


In [ ]:

# ============================================================
# IMAGE TRANSFORMS
# ============================================================

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.85, 1.0),
        ratio=(0.90, 1.10)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(288),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])


# ============================================================
# PYTORCH DATASET
# ============================================================

class DriverDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, int(row["label"])


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = DriverDataset(
    train_df,
    train_transform
)

val_dataset = DriverDataset(
    val_df,
    val_transform
)


# ============================================================
# DATA LOADER REPRODUCIBILITY
# ============================================================

def seed_worker(worker_id):

    worker_seed = torch.initial_seed() % (2**32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)


# ============================================================
# DATA LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Training samples: 19728
Validation samples: 4933
Training batches: 617
Validation batches: 155


In [ ]:

# ============================================================
# LOAD PRETRAINED MOBILENETV3-SMALL
# ============================================================

weights = models.MobileNet_V3_Small_Weights.DEFAULT

model = models.mobilenet_v3_small(
    weights=weights
)

# Replace final layer with nine independent outputs.
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(DEVICE)


# ============================================================
# CLASS WEIGHTS
# ============================================================

class_counts = np.bincount(
    train_df["label"].to_numpy(),
    minlength=NUM_CLASSES
)

if np.any(class_counts == 0):
    raise RuntimeError("A class is missing from training.")

class_weights = len(train_df) / (
    NUM_CLASSES * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)


# ============================================================
# OPTIMIZER
# ============================================================

backbone_parameters = [
    parameter
    for name, parameter in model.named_parameters()
    if not name.startswith("classifier.")
]

classifier_parameters = list(
    model.classifier.parameters()
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_parameters,
            "lr": 0.0001
        },
        {
            "params": classifier_parameters,
            "lr": 0.0005
        }
    ],
    weight_decay=0.0001
)


# ============================================================
# SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


# ============================================================
# MIXED PRECISION
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=DEVICE.type == "cuda"
)


# ============================================================
# VERIFY MODEL AND DATA LOADERS
# ============================================================

model.eval()

with torch.no_grad():

    sample_images, sample_labels = next(iter(val_loader))

    sample_images = sample_images.to(DEVICE)

    sample_outputs = model(sample_images)

assert sample_outputs.shape == (
    len(sample_images),
    NUM_CLASSES
)

assert torch.isfinite(sample_outputs).all()

model.train()

print("\n" + "=" * 60)
print("MODEL INITIALIZATION SUCCESSFUL")
print("=" * 60)

print("Architecture: MobileNetV3-Small")
print("Independent classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)
print("Output shape:", tuple(sample_outputs.shape))
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Device:", DEVICE)

print("=" * 60)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 45.5MB/s]



MODEL INITIALIZATION SUCCESSFUL
Architecture: MobileNetV3-Small
Independent classes: 9
Class names: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8']
Output shape: (32, 9)
Training images: 19728
Validation images: 4933
Device: cpu


In [ ]:

# ============================================================
# STEP 7: TRAIN MOBILE NET V3
# ============================================================

BEST_MODEL_PATH = OUTPUT_DIR / "best_driver_behavior_model.pth"

HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
BEST_METRICS_PATH = OUTPUT_DIR / "best_per_class_metrics.csv"

best_macro_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
history = []


# ============================================================
# TRAIN / VALIDATION FUNCTION
# ============================================================

def run_epoch(model, loader, training=False):

    model.train(training)

    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in tqdm(loader, leave=False):

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.detach().cpu().tolist())
        all_predictions.extend(predictions.detach().cpu().tolist())

    epoch_loss = running_loss / len(loader.dataset)

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            labels=list(range(NUM_CLASSES)),
            zero_division=0
        )
    )

    return {
        "loss": epoch_loss,
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0
        ),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
        "labels": all_labels,
        "predictions": all_predictions
    }


# ============================================================
# TRAINING LOOP
# ============================================================

start_time = time.time()

print("Starting training...")
print("Classes:", CLASS_NAMES)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Device:", DEVICE)

for epoch in range(1, EPOCHS + 1):

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 60)

    train_metrics = run_epoch(
        model, train_loader, training=True
    )

    val_metrics = run_epoch(
        model, val_loader, training=False
    )

    scheduler.step(val_metrics["loss"])

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Train loss:       {train_metrics['loss']:.4f}")
    print(f"Train accuracy:   {train_metrics['accuracy']:.4f}")
    print(f"Validation loss:  {val_metrics['loss']:.4f}")
    print(f"Validation acc:   {val_metrics['accuracy']:.4f}")
    print(f"Validation Macro-F1: {val_metrics['macro_f1']:.4f}")
    print(f"Learning rate:    {current_lr:.7f}")

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "validation_loss": val_metrics["loss"],
        "validation_accuracy": val_metrics["accuracy"],
        "validation_macro_f1": val_metrics["macro_f1"],
        "learning_rate": current_lr
    })

    pd.DataFrame(history).to_csv(
        HISTORY_PATH, index=False
    )

    # Save best checkpoint based on validation Macro-F1.
    if val_metrics["macro_f1"] > best_macro_f1:

        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "class_names": CLASS_NAMES,
            "class_to_idx": CLASS_TO_IDX,
            "num_classes": NUM_CLASSES,
            "image_size": IMAGE_SIZE,
            "imagenet_mean": IMAGENET_MEAN,
            "imagenet_std": IMAGENET_STD,
            "best_macro_f1": best_macro_f1,
            "validation_loss": val_metrics["loss"],
            "validation_accuracy": val_metrics["accuracy"]
        }, BEST_MODEL_PATH)

        pd.DataFrame({
            "class_name": CLASS_NAMES,
            "precision": val_metrics["precision"],
            "recall": val_metrics["recall"],
            "f1_score": val_metrics["f1"],
            "support": val_metrics["support"]
        }).to_csv(BEST_METRICS_PATH, index=False)

        print("Best checkpoint saved.")

    else:
        epochs_without_improvement += 1
        print(
            f"No improvement: {epochs_without_improvement}/{PATIENCE}"
        )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

assert checkpoint["class_names"] == CLASS_NAMES
assert checkpoint["num_classes"] == NUM_CLASSES

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nTRAINING FINISHED")
print("Best epoch:", checkpoint["epoch"])
print("Best validation Macro-F1:", checkpoint["best_macro_f1"])
print("Best validation accuracy:", checkpoint["validation_accuracy"])
print("Training time (minutes):", round((time.time() - start_time) / 60, 2))
print("Checkpoint:", BEST_MODEL_PATH)


Starting training...
Classes: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8']
Training images: 19728
Validation images: 4933
Device: cpu

Epoch 1/25
------------------------------------------------------------


  0%|          | 0/617 [00:00<?, ?it/s]

  0%|          | 0/155 [00:00<?, ?it/s]

Train loss:       0.6723
Train accuracy:   0.8732
Validation loss:  0.4699
Validation acc:   0.9491
Validation Macro-F1: 0.9480
Learning rate:    0.0001000
Best checkpoint saved.

Epoch 2/25
------------------------------------------------------------


  0%|          | 0/617 [00:00<?, ?it/s]

  0%|          | 0/155 [00:00<?, ?it/s]

... (18 outputs left out)

Train loss:       0.3057
Train accuracy:   0.9936
Validation loss:  0.3907
Validation acc:   0.9659
Validation Macro-F1: 0.9654
Learning rate:    0.0001000
No improvement: 4/5

Epoch 9/25
------------------------------------------------------------


  0%|          | 0/617 [00:00<?, ?it/s]

  0%|          | 0/155 [00:00<?, ?it/s]

Train loss:       0.3043
Train accuracy:   0.9936
Validation loss:  0.3888
Validation acc:   0.9670
Validation Macro-F1: 0.9665
Learning rate:    0.0001000
No improvement: 5/5
Early stopping.

TRAINING FINISHED
Best epoch: 4
Best validation Macro-F1: 0.9723921320779314
Best validation accuracy: 0.97263328603284
Training time (minutes): 188.49
Checkpoint: /content/driver_model_project/outputs/best_driver_behavior_model.pth


In [ ]:

# ============================================================
# FINAL VALIDATION EVALUATION
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

final_metrics = run_epoch(
    model,
    val_loader,
    training=False
)

print("\nVALIDATION RESULTS")
print("Accuracy:", round(final_metrics["accuracy"], 4))
print("Macro-F1:", round(final_metrics["macro_f1"], 4))

report = classification_report(
    final_metrics["labels"],
    final_metrics["predictions"],
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
display(report_df.round(4))

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

per_class_df = pd.DataFrame({
    "class_name": CLASS_NAMES,
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "f1_score": final_metrics["f1"],
    "support": final_metrics["support"]
})

display(per_class_df.round(4))

per_class_df.to_csv(
    OUTPUT_DIR / "per_class_metrics.csv",
    index=False
)

cm = confusion_matrix(
    final_metrics["labels"],
    final_metrics["predictions"],
    labels=list(range(NUM_CLASSES))
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print("Confusion matrix:")
display(cm_df)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

cm_norm = confusion_matrix(
    final_metrics["labels"],
    final_metrics["predictions"],
    labels=list(range(NUM_CLASSES)),
    normalize="true"
)

pd.DataFrame(
    cm_norm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    OUTPUT_DIR / "confusion_matrix_normalized.csv"
)

print("\nValidation evaluation saved.")


  0%|          | 0/155 [00:00<?, ?it/s]


VALIDATION RESULTS
Accuracy: 0.9726
Macro-F1: 0.9724


,precision,recall,f1-score,support
c0,0.9648,0.9133,0.9384,600.0000
c1,0.9773,0.9756,0.9764,573.0000
c2,0.9805,0.9805,0.9805,565.0000
c3,0.9857,0.9928,0.9893,557.0000
c4,0.9946,0.9946,0.9946,555.0000
c5,0.9608,0.9895,0.9749,570.0000
c6,0.9762,0.9798,0.9780,545.0000
c7,0.9737,0.9877,0.9807,488.0000
c8,0.9358,0.9417,0.9387,480.0000
accuracy,0.9726,0.9726,0.9726,0.9726


,class_name,precision,recall,f1_score,support
0,c0,0.9648,0.9133,0.9384,600
1,c1,0.9773,0.9756,0.9764,573
2,c2,0.9805,0.9805,0.9805,565
3,c3,0.9857,0.9928,0.9893,557
4,c4,0.9946,0.9946,0.9946,555
5,c5,0.9608,0.9895,0.9749,570
6,c6,0.9762,0.9798,0.9780,545
7,c7,0.9737,0.9877,0.9807,488
8,c8,0.9358,0.9417,0.9387,480


Confusion matrix:


,c0,c1,c2,c3,c4,c5,c6,c7,c8
c0,548,1,4,2,2,18,6,0,19
c1,2,559,5,2,1,1,2,0,1
c2,2,4,554,0,0,0,3,1,1
c3,2,1,0,553,0,1,0,0,0
c4,1,0,0,2,552,0,0,0,0
c5,2,0,1,0,0,564,0,2,1
c6,2,5,1,0,0,0,534,0,3
c7,0,0,0,0,0,0,0,482,6
c8,9,2,0,2,0,3,2,10,452



Validation evaluation saved.


In [ ]:

TEST_DIR = TRAIN_DIR.parent / "Test"

assert TEST_DIR.exists(), f"Test folder not found: {TEST_DIR}"

print("Test folder:", TEST_DIR)
print("Test images:", sum(
    1 for p in TEST_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
))


Test folder: /content/driver_model_project/downloads/alternative_driver/New_Driver_Behaviour2/Test
Test images: 6256


In [ ]:

test_class_dirs = {
    p.name: p
    for p in TEST_DIR.iterdir()
    if p.is_dir()
}

has_labels = all(name in test_class_dirs for name in CLASS_NAMES)

test_records = []

if has_labels:
    print("Labelled test folders detected.")

    for class_name in CLASS_NAMES:
        for path in test_class_dirs[class_name].rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                test_records.append({
                    "path": str(path),
                    "label": CLASS_TO_IDX[class_name]
                })
else:
    print("Test folder has no complete set of class labels.")

    for path in TEST_DIR.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            test_records.append({"path": str(path)})

if not test_records:
    raise RuntimeError("No test images found.")

print("Total test images:", len(test_records))


class OriginalTestDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        with Image.open(self.records[index]["path"]) as image:
            image = image.convert("RGB")
            image = val_transform(image)

        return image


test_dataset = OriginalTestDataset(test_records)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


model.eval()

test_predictions = []
test_confidences = []
test_probabilities = []

with torch.no_grad():
    for images in tqdm(test_loader):
        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        confidence, predictions = probabilities.max(dim=1)

        test_predictions.extend(predictions.cpu().tolist())
        test_confidences.extend(confidence.cpu().tolist())
        test_probabilities.extend(probabilities.cpu().tolist())


test_results = pd.DataFrame({
    "image_path": [r["path"] for r in test_records],
    "predicted_index": test_predictions,
    "predicted_class": [IDX_TO_CLASS[i] for i in test_predictions],
    "confidence": test_confidences
})

for index, class_name in enumerate(CLASS_NAMES):
    test_results[f"probability_{class_name}"] = [
        row[index] for row in test_probabilities
    ]

test_results.to_csv(
    OUTPUT_DIR / "original_test_predictions.csv",
    index=False
)

print("\nTEST PREDICTIONS GENERATED")
print(test_results.head(10))

if has_labels:
    true_labels = [r["label"] for r in test_records]

    print("\nTEST ACCURACY:",
          accuracy_score(true_labels, test_predictions))

    print("\nTEST CLASSIFICATION REPORT:")

    print(classification_report(
        true_labels,
        test_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        zero_division=0
    ))

else:
    print("\nNo ground-truth labels available.")
    print("Independent test accuracy cannot be calculated.")

print("\nTest results saved successfully.")


Test folder has no complete set of class labels.
Total test images: 6256


  0%|          | 0/196 [00:00<?, ?it/s]


TEST PREDICTIONS GENERATED
                                          image_path  predicted_index  \
0  /content/driver_model_project/downloads/altern...                6   
1  /content/driver_model_project/downloads/altern...                6   
2  /content/driver_model_project/downloads/altern...                6   
3  /content/driver_model_project/downloads/altern...                6   
4  /content/driver_model_project/downloads/altern...                6   
5  /content/driver_model_project/downloads/altern...                6   
6  /content/driver_model_project/downloads/altern...                6   
7  /content/driver_model_project/downloads/altern...                6   
8  /content/driver_model_project/downloads/altern...                6   
9  /content/driver_model_project/downloads/altern...                6   

  predicted_class  confidence  probability_c0  probability_c1  probability_c2  \
0              c6    0.719819        0.026033        0.175409        0.014749   
1     

In [ ]:

# ============================================================
# EXPORT MOBILENETV3 TO ONNX
# ============================================================

import onnx
import onnxruntime as ort

ONNX_PATH = OUTPUT_DIR / "driver_behavior_mobilenetv3.onnx"

model.eval()

dummy_input = torch.randn(
    1, 3, IMAGE_SIZE, IMAGE_SIZE,
    device=DEVICE
)

torch.onnx.export(
    model,
    dummy_input,
    str(ONNX_PATH),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "logits": {0: "batch_size"}
    },
    dynamo=False
)

# Check ONNX structure.
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

# Load ONNX Runtime.
session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

# Compare PyTorch and ONNX outputs.
max_difference = 0.0

for batch_size in [1, 2, 4]:

    test_input = torch.randn(
        batch_size,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        device=DEVICE
    )

    with torch.no_grad():
        pytorch_output = model(test_input).cpu().numpy()

    onnx_output = session.run(
        ["logits"],
        {"input": test_input.cpu().numpy()}
    )[0]

    difference = float(
        np.max(np.abs(pytorch_output - onnx_output))
    )

    max_difference = max(max_difference, difference)

    assert pytorch_output.shape == onnx_output.shape
    assert np.array_equal(
        pytorch_output.argmax(axis=1),
        onnx_output.argmax(axis=1)
    ), "PyTorch and ONNX predictions do not match."

    print(
        f"Batch {batch_size}: maximum difference = {difference:.8f}"
    )

    if difference >= 1e-4:
        raise RuntimeError("ONNX numerical validation failed.")

print("\nONNX validation successful!")
print("Output classes:", NUM_CLASSES)


# ============================================================
# SAVE FINAL METADATA
# ============================================================

metadata = {
    "model": "MobileNetV3-Small",
    "format": "ONNX",
    "opset_version": 17,
    "image_size": IMAGE_SIZE,
    "input_name": "input",
    "output_name": "logits",
    "input_shape": [None, 3, IMAGE_SIZE, IMAGE_SIZE],
    "output_shape": [None, NUM_CLASSES],
    "input_format": "RGB",
    "normalization_mean": list(IMAGENET_MEAN),
    "normalization_std": list(IMAGENET_STD),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "num_classes": NUM_CLASSES,
    "best_epoch": int(checkpoint["epoch"]),
    "validation_accuracy": float(final_metrics["accuracy"]),
    "validation_macro_f1": float(final_metrics["macro_f1"]),
    "maximum_onnx_output_difference": max_difference
}

with open(
    OUTPUT_DIR / "model_metadata.json",
    "w"
) as file:
    json.dump(metadata, file, indent=4)

print("Metadata saved.")


/tmp/ipykernel_481/1539177970.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Batch 1: maximum difference = 0.00000186
Batch 2: maximum difference = 0.00000270
Batch 4: maximum difference = 0.00000203

ONNX validation successful!
Output classes: 9
Metadata saved.


In [ ]:
from google.colab import files

files.download(str(ONNX_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>